In [4]:
import numpy as np
import pandas as pd
import time
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import log_loss, roc_auc_score, accuracy_score, mean_absolute_error
from src import util
plays_df = pd.read_csv('data/pbp/pbp_2005_2025_plays.csv')
drives_df = pd.read_csv('data/pbp/pbp_2005_2025_drives.csv')

In [ ]:
# ============================================================
# Covariate-dependent Markov, bin-based features (state_id-consistent)
#   baseline    : state (down, ytg_bin, field_bin)
#   +time       : state + game_seconds_remaining
#   +time+score : state + time + score_differential
# Play-level + Drive-level simulation (greedy & sample).
# Features derived from state_id so simulation reconstruction is exact.
# ============================================================

SPLIT_MODE = 'cut2020' # pick 4 train/test splits - 'cut2020', 'all', 'bundle', 'yearly'
MIN_NEXT_COUNT = 300 # exclude rare 'next_states' 
FIRST_DOWN_STATES = list(range(1, 61))
ABS_IDS = list(util.ABSORBING_STATES_MAP.keys())
ABS_POINTS = {241: 7, 242: 3, 243: 0, 244: 0, 245: 0, 246: 0} # for calculating drive MAE
MAX_SIM_PLAYS = 40
N_SIM_DRIVES = 3000            # only used if sampling is enabled below
RNG = np.random.RandomState(42)

train_list, test_list, test_drives_list = util.get_train_test_sets(
    plays_df, drives_df, split_mode=SPLIT_MODE, train_frac=0.75)

_, ydstogo_labels = util.get_ydstogo_cuts_labels()
_, field_labels = util.get_field_position_cuts_labels()
state_values = util.get_state_values(
    ["down", "binned_ydstogo_id", "binned_field_position_id"],
    [[1, 2, 3, 4], ydstogo_labels, field_labels])
_, id_to_state, _ = util.generate_state_ids(state_values, util.ABSORBING_STATES)

N_YTG = len(ydstogo_labels)
N_FIELD = len(field_labels)

def make_features(state_ids, score_diff, gsr, use_time, use_score):
    downs = np.array([id_to_state[s][0] for s in state_ids])
    ytgs = np.array([id_to_state[s][1] for s in state_ids])
    fields = np.array([id_to_state[s][2] for s in state_ids])
    cols = []
    for dn in [1, 2, 3, 4]:
        cols.append((downs == dn).astype(float))
    for y in range(1, N_YTG + 1):
        cols.append((ytgs == y).astype(float))
    for f in range(1, N_FIELD + 1):
        cols.append((fields == f).astype(float))
    X = np.column_stack(cols)
    if use_time:
        X = np.column_stack([X, np.asarray(gsr, float)])
    if use_score:
        X = np.column_stack([X, np.asarray(score_diff, float)])
    return X

MODELS = {
    'baseline':    dict(use_time=False, use_score=False),
    '+time':       dict(use_time=True,  use_score=False),
    '+time+score': dict(use_time=True,  use_score=True),
}

REQ = ['cur_state_id', 'next_state_id', 'score_differential', 'game_seconds_remaining']

def top_k_acc(proba, y_true, classes, k):
    topk = np.argsort(proba, axis=1)[:, -k:]
    ca = np.array(classes)
    return np.mean([y_true[i] in ca[topk[i]] for i in range(len(y_true))])

def eval_fold(train_df, test_df, test_drives_df):
    tr = train_df.dropna(subset=REQ).copy()
    te = test_df.dropna(subset=REQ).copy()
    tr = tr[tr['cur_state_id'].isin(id_to_state.keys())]
    te = te[te['cur_state_id'].isin(id_to_state.keys())]
    vc = tr['next_state_id'].value_counts()
    keep = set(vc[vc >= MIN_NEXT_COUNT].index)
    tr = tr[tr['next_state_id'].isin(keep)]
    te = te[te['next_state_id'].isin(keep)]
    print(f"    train={len(tr):,} test={len(te):,} classes={len(keep)}", flush=True)

    y_tr = tr['next_state_id'].values
    y_te = te['next_state_id'].values
    tr_states = tr['cur_state_id'].values.astype(int)
    te_states = te['cur_state_id'].values.astype(int)
    tr_sd, tr_gsr = tr['score_differential'].values, tr['game_seconds_remaining'].values
    te_sd, te_gsr = te['score_differential'].values, te['game_seconds_remaining'].values

    dd = test_drives_df.dropna(subset=['start_state_id', 'drive_result_id']).copy()
    dd = dd[dd['start_state_id'].isin([s for s in id_to_state if s <= 240])]

    fold_res = {}
    for name, cfg in MODELS.items():
        t0 = time.time()
        print(f"    fitting {name} ...", flush=True)
        X_tr = make_features(tr_states, tr_sd, tr_gsr, **cfg)
        scaler = StandardScaler().fit(X_tr)
        X_tr = scaler.transform(X_tr)
        clf = LogisticRegression(max_iter=1000, solver='saga', tol=1e-3)
        clf.fit(X_tr, y_tr)
        classes = clf.classes_

        # ---- play-level ----
        X_te = scaler.transform(make_features(te_states, te_sd, te_gsr, **cfg))
        proba = clf.predict_proba(X_te)
        ll = log_loss(y_te, proba, labels=classes)
        t5 = top_k_acc(proba, y_te, classes, 5)
        t10 = top_k_acc(proba, y_te, classes, 10)
        fd_cols = [i for i, c in enumerate(classes) if c in FIRST_DOWN_STATES]
        fd_prob = proba[:, fd_cols].sum(axis=1)
        fd_true = np.isin(y_te, FIRST_DOWN_STATES).astype(int)
        fd_auc = roc_auc_score(fd_true, fd_prob) if 0 < fd_true.sum() < len(fd_true) else np.nan

        # ---- drive-level simulation ----
        abs_set = set(ABS_IDS)

        # ===== toggle: FULL vs SAMPLED =====
        sample_drives = dd                                              # FULL (all drives)
        # sample_drives = dd.sample(n=min(N_SIM_DRIVES, len(dd)), random_state=1)  # SAMPLED

        print(f"    simulating {len(sample_drives):,} drives ({name}) ...", flush=True)
        s0_arr = sample_drives['start_state_id'].values.astype(int)
        sd_arr = sample_drives['start_score_differential'].fillna(0).values
        gsr_arr = sample_drives['start_time_rem'].fillna(1800).values
        actual_arr = sample_drives['drive_result_id'].values.astype(int)

        def simulate(s0, sd, gsr, mode):
            state = s0
            for _ in range(MAX_SIM_PLAYS):
                if state in abs_set:
                    return state
                x = scaler.transform(make_features([state], [sd], [gsr], **cfg))
                p = clf.predict_proba(x)[0]
                state = int(classes[np.argmax(p)]) if mode == 'greedy' \
                        else int(RNG.choice(classes, p=p))
            return state if state in abs_set else 246

        for mode in ['greedy', 'sample']:
            preds = np.empty(len(s0_arr), dtype=int)
            for i in range(len(s0_arr)):
                preds[i] = simulate(s0_arr[i], sd_arr[i], gsr_arr[i], mode)
                if (i + 1) % 5000 == 0:
                    print(f"      {name} {mode}: {i+1:,}/{len(s0_arr):,}", flush=True)
            fold_res.setdefault(name, {})
            fold_res[name][f'drive_final_acc_{mode}'] = accuracy_score(actual_arr, preds)
            fold_res[name][f'drive_score_acc_{mode}'] = accuracy_score(
                np.isin(actual_arr, [241, 242]), np.isin(preds, [241, 242]))
            fold_res[name][f'drive_td_acc_{mode}'] = accuracy_score(
                actual_arr == 241, preds == 241)
            fold_res[name][f'drive_points_mae_{mode}'] = mean_absolute_error(
                [ABS_POINTS.get(a, 0) for a in actual_arr],
                [ABS_POINTS.get(p, 0) for p in preds])

        fold_res[name].update({'state_log_loss': ll, 'top5_acc': t5,
                               'top10_acc': t10, 'first_down_roc_auc': fd_auc})
        print(f"    {name} done in {time.time()-t0:.1f}s (ll={ll:.3f})", flush=True)
    return fold_res

print(f"split_mode={SPLIT_MODE}, folds={len(train_list)}", flush=True)
all_folds = []
for k, (a, b, c) in enumerate(zip(train_list, test_list, test_drives_list)):
    print(f"  fold {k+1}/{len(train_list)}", flush=True)
    all_folds.append(eval_fold(a, b, c))

models = list(MODELS.keys())
mets = sorted({m for f in all_folds for md in f.values() for m in md})
summary = pd.DataFrame(
    {mdl: {mt: np.nanmean([f[mdl][mt] for f in all_folds if mt in f[mdl]])
           for mt in mets} for mdl in models}).T

play_cols = ['state_log_loss', 'top5_acc', 'top10_acc', 'first_down_roc_auc']
drive_cols = [c for c in mets if c.startswith('drive_')]

print("\n" + "=" * 75)
print(f"PLAY-LEVEL (split={SPLIT_MODE})")
print("=" * 75)
print(summary[play_cols].round(4).to_string())
print("\n" + "=" * 75)
print("DRIVE-LEVEL (simulation)")
print("=" * 75)
print(summary[drive_cols].round(4).to_string())


Getting train/test sets using split mode "cut2020" with training fraction of games = 0.75
split_mode=cut2020, folds=1
  fold 1/1
    train=601,105 test=196,066 classes=200
    fitting baseline ...
